In [4]:
from pathlib import Path

import gc

In [5]:
import numpy as np
import xarray as xr

In [6]:
from data import ds_single, ds_pressure, _load_single_dataset

In [7]:
from compression_recommendations import Recommendations
from compression_recommendations.filters.abc import Filter
from compression_recommendations.filters.cf import (
    CfShortNameFilter,
    CfStandardNameFilter,
)
from compression_recommendations.filters.combinators import AllFilters, AnyFilter
from compression_recommendations.filters.grib import GribShortNameFilter
from compression_recommendations.filters.level import (
    LevelKind,
    LevelKindFilter,
    LevelValueFilter,
)
from compression_recommendations.filters.tag import TagFilter
from compression_recommendations.recommendation import Recommendation
from compression_recommendations.requirements.error_bounds.max import (
    MaxPointwiseAbsoluteErrorBoundRequirement,
    MaxPointwiseRelativeErrorBoundRequirement,
    MaxPointwiseQuadraticErrorBoundRequirement,
)
from semver import Version

In [20]:
def compute_ensemble_spread_recommendations(
    mean: xr.DataArray, spread: xr.DataArray, percentiles: list[float], extra_filters: list[Filter],
) -> list[Recommendation]:
    da = mean
    
    mean = mean.values.flatten()
    spread = spread.values.flatten()

    spread_nonzero = spread[spread > 0.0]

    # compute the absolute error bound
    if len(spread_nonzero) > 0:
        absolute = np.nanquantile(spread_nonzero, [1 - p for p in percentiles])
    else:
        absolute = [0.0 for _ in percentiles]

    # compute the relative error bound
    abs_mean = np.abs(mean)
    rel = spread[abs_mean > 0.0] / abs_mean[abs_mean > 0.0]
    rel_nonzero = rel[rel > 0.0]

    if len(rel_nonzero) > 0:
        relative = np.nanquantile(rel_nonzero, [1 - p for p in percentiles])
    else:
        relative = [0.0 for _ in percentiles]

    amin, amax = np.nanmin(mean), np.nanmax(mean)

    # compute the quadratic error bound
    # if we map [amin, amax] to [-1, 1], then fitting a quadratic to
    #  (-1, 0), (x', y'), (1, 0) will reduce to
    # f(x) = a - a*x^2
    #  so we can fit a by using
    # y' = a - a*(x')^2
    # y' = a * (1 - (x')^2)
    # a = (y') / (1 - (x')^2)
    with np.errstate(divide="ignore", over="ignore", under="ignore", invalid="ignore"):
        a = spread / (1.0 - (-1.0 + 2.0 * (mean - amin) / (amax - amin)) ** 2)
    a_nonzero = a[a > 0.0]

    if len(a_nonzero) > 0:
        quadratic = np.nanquantile(a_nonzero, [1 - p for p in percentiles])
    else:
        quadratic = [0.0 for _ in percentiles]

    # compute a histogram of the mean-spread pairs with bins across the mean
    nbins = 100
    mean_bin_edges = np.nanquantile(mean, np.linspace(0.0, 1.0, nbins + 1))
    mean_bin_centres = (mean_bin_edges[:-1] + mean_bin_edges[1:]) / 2
    mean_bin_counts, _ = np.histogram(mean, bins=mean_bin_edges)
    ibin = np.minimum(np.searchsorted(mean_bin_edges, mean), nbins - 1)

    picks = []
    for j, p in enumerate(percentiles):
        # compute each bin's spread quantile
        mean_bin_spread_bounds = np.zeros(nbins)
        for i in range(nbins):
            ispread = spread[ibin == i]
            if len(ispread) > 0:
                bs = np.nanquantile(ispread, 1-p)
            else:
                bs = np.nan
            mean_bin_spread_bounds[i] = bs

        # approximate the "cost" of each error bound as the distance between
        # the binned spread and the error bound
        cost_abs = np.nansum(np.abs(mean_bin_spread_bounds - absolute[j]) * mean_bin_counts) / np.sum(mean_bin_counts)
        cost_rel = np.nansum(np.abs(mean_bin_spread_bounds - np.abs(mean_bin_centres) * relative[j]) * mean_bin_counts) / np.sum(mean_bin_counts)
        cost_qua = np.nansum(np.abs(mean_bin_spread_bounds - (quadratic[j] - quadratic[j] * np.square(-1.0 + 2.0 * (mean_bin_centres - amin) / (amax - amin)))) * mean_bin_counts) / np.sum(mean_bin_counts)

        # pick the error bound with the smallest cost
        costs = [("absolute", cost_abs), ("relative", cost_rel), ("quadratic", cost_qua)]
        picks.append(sorted(costs, key=lambda x: x[1])[0][0])

    base_filters = [
        AnyFilter(
            filters=(
                [CfStandardNameFilter(value=da.standard_name)]
                if da.standard_name.lower() != "unknown"
                else []
            )
            + [
                CfShortNameFilter(value=da.name),
                GribShortNameFilter(value=da.GRIB_shortName),
            ]
        ),
    ]

    recommendations = []

    # compile the recommendations for each percentile and error bound kind
    for i, p in enumerate(percentiles):
        filters = base_filters + [TagFilter(value=f"{p * 100}%")] + extra_filters

        recommendations.append(
            Recommendation(
                filters=filters + [AnyFilter(filters=[
                    TagFilter(value="absolute")
                ] + ([TagFilter(value="pick")] if picks[i] == "absolute" else []))],
                requirements=[MaxPointwiseAbsoluteErrorBoundRequirement(value=absolute[i])],
            )
        )
        recommendations.append(
            Recommendation(
                filters=filters + [AnyFilter(filters=[
                    TagFilter(value="relative")
                ] + ([TagFilter(value="pick")] if picks[i] == "relative" else []))],
                requirements=[MaxPointwiseRelativeErrorBoundRequirement(value=relative[i])],
            )
        )
        recommendations.append(
            Recommendation(
                filters=filters + [AnyFilter(filters=[
                    TagFilter(value="quadratic")
                ] + ([TagFilter(value="pick")] if picks[i] == "quadratic" else []))],
                requirements=[MaxPointwiseQuadraticErrorBoundRequirement(value=quadratic[i], minimum=amin, maximum=amax)],
            )
        )

    return recommendations

In [21]:
from tqdm import tqdm

In [22]:
recommendations = []

In [23]:
for i, key in tqdm(enumerate(sorted(ds_single.mean))):
    if (i > 0) and (i % 50 == 0):
        del ds_single
        gc.collect()
        ds_single = _load_single_dataset()
    recommendations += compute_ensemble_spread_recommendations(
        mean=ds_single.mean[key],
        spread=ds_single.spread[key],
        percentiles=[1.0, 0.999, 0.99, 0.95],
        extra_filters=[
            AllFilters(
                filters=[
                    LevelKindFilter(value=LevelKind.single),
                ]
            )
        ],
    )

/Users/junityre/era5-ensemble/.venv/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1573: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(
226it [3:35:45, 57.28s/it]


In [ ]:
for key in tqdm(sorted(ds_pressure.mean)):
    for pressure in sorted(ds_pressure.mean[key].pressure_level.values):
        recommendations += compute_ensemble_spread_recommendations(
            mean=ds_pressure.mean[key].sel(pressure_level=pressure),
            spread=ds_pressure.spread[key].sel(pressure_level=pressure),
            percentiles=[1.0, 0.999, 0.99, 0.95],
            extra_filters=[
                AllFilters(
                    filters=[
                        LevelKindFilter(value=LevelKind.pressure),
                        LevelValueFilter(minimum=pressure, maximum=pressure),
                    ]
                )
            ],
        )

/var/folders/8v/swxsmn0d4vz5yzwjhf6bc26x3g7lq6/T/ipykernel_66419/3201369358.py:69: RuntimeWarning: invalid value encountered in divide49:51, 213.69s/it]
  cost_qua = np.nansum(np.abs(mean_bin_spread_bounds - (quadratic[j] - quadratic[j] * np.square(-1.0 + 2.0 * (mean_bin_centres - amin) / (amax - amin)))) * mean_bin_counts) / np.sum(mean_bin_counts)
 75%|█████████████████████████████████████████████████████████████████████████████████████▌                            | 12/16 [59:06<24:20, 365.06s/it]

In [ ]:
with Path("recommendations").joinpath("ensemble-spread.yaml").open("w") as f:
    Recommendations(
        recommendations=recommendations,
        version=Version.parse("0.1.0"),
        metadata={"commit": ""},  # FIXME
    ).dump(f)